# Initialization

In [0]:
%run ../../utils/config

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")
df.limit(5).display()

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
      df = df.withColumn(field.name, trim(col(field.name)))

df.limit(5).display()

## Product Key Parsing

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

In [0]:
df.limit(2).display()

## Cost Cleanup

In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

In [0]:
df.limit(2).display()

## Product Line Normalization

In [0]:
df = (
    df
    # Normalize product line
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

In [0]:
df.limit(2).display()

## Date Casting

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))
df.limit(2).display()

## Renaming Columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe

In [0]:
df.limit(5).display()

# Writing Silver Table

In [0]:
from delta.tables import DeltaTable 

TARGET_TABLE = TABLES['crm_prd']
PK_COL = "product_id"

if not spark.catalog.tableExists(TARGET_TABLE):
    df.write.format("delta").saveAsTable(TARGET_TABLE)
else:
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)
    (
        delta_target.alias("target")
        .merge(
            df.alias("source"),
            f"target.{PK_COL} = source.{PK_COL}"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# Collect MERGE Metrics

In [0]:
dt = DeltaTable.forName(spark, TARGET_TABLE)
metrics = dt.history(1).select("operationMetrics").collect()[0]
op_metrics = metrics["operationMetrics"]

# Sanity checks of silver table

In [0]:
spark.sql(f"SELECT * FROM {TARGET_TABLE} LIMIT 10").display()

# Data Quality Checks

In [0]:
df_check = spark.table(TARGET_TABLE)
total_rows = df_check.count()
null_product_id = df_check.filter(F.col(PK_COL).isNull()).count()
duplicate_rows = total_rows - df_check.select(PK_COL).distinct().count()

print(f"[QC] {TARGET_TABLE}")
print(f"  Total rows              : {total_rows}")
print(f"  Null product_id         : {null_product_id}")
print(f"  Duplicate (id) rows: {duplicate_rows}")

try:
    assert total_rows      > 0,  f"[QC FAILED] {TARGET_TABLE} is empty"
    assert null_product_id == 0, f"[QC FAILED] {null_product_id} null product_id values"
    assert duplicate_rows  == 0, f"[QC FAILED] {duplicate_rows} duplicate (product_id) rows"
    qc_status = "PASS"
    qc_message = "All checks passed"
    print("[QC PASSED]")
except AssertionError as e:
    qc_status = "FAIL"
    qc_message = str(e)
    print(f"[QC FAILED] {qc_message}")

## Write Audit Log

In [0]:
from datetime import datetime

row = [{
    "notebook_name": "silver_crm_prd_info",
    "target_table": TARGET_TABLE,
    "run_timestamp": datetime.now(),
    "rows_inserted": int(op_metrics.get("numTargetRowsInserted", 0)),
    "rows_updated": int(op_metrics.get("numTargetRowsUpdated", 0)),
    "rows_deleted": int(op_metrics.get("numTargetRowsDeleted", 0)),
    "qc_status": qc_status,
    "qc_message": qc_message
}]

df_audit = spark.createDataFrame(row)
df_audit.write.mode("append").format("delta").saveAsTable(TABLES["audit_log"])

## Sanity Check - Audit Log

In [0]:
spark.sql(f"SELECT * FROM {TABLES["audit_log"]}").display()